In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

from xgboost import XGBClassifier

In [3]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB 660.6 kB/s eta 0:02:34
   ---------------------------------------- 0.1/101.7 MB 825.8 kB/s eta 0:02:04
   ---------------------------------------- 0.1/101.7 MB 438.9 kB/s eta 0:03:52
   ---------------------------------------- 0.1/101.7 MB 525.1 kB/s eta 0:03:14
   ---------------------------------------- 0.1/101.7 MB 554.9 kB/s eta 0:03:04
   ---------------------------------------- 0.1/101.7 MB 554.9 kB/s eta 0:03:04
   ---------------------------------------- 0.1/101.7 MB 448.2 kB/s eta 0:03:47
   ---------------------------------------- 0.2/101.7 MB 418.0 kB/s eta 0:04:03
   ---------------------------------------- 0.2/101.7 MB 418.0 kB/s eta 0:04:03
   ---------------------------------------- 0.2/101.7 MB 418.0 kB/s eta 0:04:03
   ---------------------------------------- 0.2/101.7 MB 4

In [14]:
df = pd.read_csv("SampleTransaction12.csv", header=None)
df = df[0].str.split(",", expand=True)

# Set header
df.columns = df.iloc[0]
df = df[1:]

df.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,None
1,11978328,2012-11-23 16:03:00,619,3348,$263.43,Swipe Transaction,54850,Austin,MN,55912.0,4814,,None
2,11363233,2012-07-07 13:57:00,456,4576,$38.26,Swipe Transaction,68135,Cape Coral,FL,33909.0,5411,,None
3,8117710,2010-06-10 16:59:00,209,4676,$52.57,Swipe Transaction,81833,El Paso,TX,79928.0,5912,,None
4,12606562,2013-04-13 12:08:00,1605,1133,$40.00,Swipe Transaction,27092,Amelia,OH,45102.0,4829,,None
5,12628171,2013-04-18 10:00:00,144,5247,$4.58,Swipe Transaction,44578,Arkadelphia,AR,71923.0,5812,,None


In [16]:
# Convert amount
df['amount'] = df['amount'].str.replace('$', '').astype(float)

# Convert date
df['date'] = pd.to_datetime(df['date'])

# Drop useless column
df = df.drop(columns=['None'], errors='ignore')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 1 to 1048575
Data columns (total 13 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   id              1048575 non-null  object        
 1   date            1048575 non-null  datetime64[ns]
 2   client_id       1048575 non-null  object        
 3   card_id         1048575 non-null  object        
 4   amount          1048575 non-null  float64       
 5   use_chip        1048575 non-null  object        
 6   merchant_id     1048575 non-null  object        
 7   merchant_city   1048575 non-null  object        
 8   merchant_state  1048575 non-null  object        
 9   zip             1048575 non-null  object        
 10  mcc             1048575 non-null  object        
 11  errors          1048575 non-null  object        
 12  None            74 non-null       object        
dtypes: datetime64[ns](1), float64(1), object(11)
memory usage: 104.0+ MB


In [20]:
import json

with open("train_fraud_labels.json") as f:
    labels = json.load(f)

labels_df = pd.DataFrame(list(labels['target'].items()), columns=['id', 'fraud'])

df['id'] = df['id'].astype(str)
labels_df['id'] = labels_df['id'].astype(str)

df = df.merge(labels_df, on='id', how='left')

In [22]:
# Time features
df['hour'] = df['date'].dt.hour
df['day'] = df['date'].dt.dayofweek

# High amount flag
df['high_amount'] = (df['amount'] > df['amount'].quantile(0.95)).astype(int)

# Error flag
df['has_error'] = df['errors'].notna().astype(int)

# MCC risk (manual)
high_risk_mcc = ['4829', '7995']
df['high_risk_mcc'] = df['mcc'].isin(high_risk_mcc).astype(int)

In [24]:
le = LabelEncoder()

df['merchant_city'] = le.fit_transform(df['merchant_city'].astype(str))
df['merchant_state'] = le.fit_transform(df['merchant_state'].astype(str))
df['use_chip'] = le.fit_transform(df['use_chip'].astype(str))

In [30]:
df['fraud'] = df['fraud'].map({'Yes':1, 'No':0})

df = df.dropna(subset=['fraud'])

In [32]:
features = [
    'amount', 'hour', 'day',
    'high_amount', 'has_error',
    'high_risk_mcc',
    'merchant_city', 'merchant_state', 'use_chip'
]

X = df[features]
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(scale_pos_weight=10)  # handle imbalance
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [34]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    140324
         1.0       0.40      0.38      0.39       224

    accuracy                           1.00    140548
   macro avg       0.70      0.69      0.69    140548
weighted avg       1.00      1.00      1.00    140548

ROC AUC: 0.6870581653886719


In [36]:
import joblib

joblib.dump(model, "fraud_model.pkl")

['fraud_model.pkl']